In [ ]:
# ULTRA-COMPACT TETRIS - Perfect for iPhone
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets
import time
import threading

# Simple Tetris implementation
class SimpleTetris:
    def __init__(self):
        self.width, self.height = 6, 10  # Even smaller board
        self.reset()
    
    def reset(self):
        self.board = np.zeros((self.height, self.width))
        self.score = 0
        self.game_over = False
        self.pieces = [I'm
            np.array([[1,1,1,1]]),           # I
            np.array([[1,1],[1,1]]),         # O  
            np.array([[1,1,1],[0,1,0]]),     # T
            np.array([[1,1,1],[1,0,0]]),     # J
            np.array([[1,1,1],[0,0,1]]),     # L
            np.array([[0,1,1],[1,1,0]]),     # S
            np.array([[1,1,0],[0,1,1]])      # Z
        ]
        self.colors = ['cyan', 'yellow', 'purple', 'blue', 'orange', 'green', 'red']
        self.new_piece()
    
    def new_piece(self):
        piece_idx = np.random.randint(len(self.pieces))
        self.piece = self.pieces[piece_idx]
        self.piece_color = self.colors[piece_idx]
        self.piece_x = self.width//2 - self.piece.shape[1]//2
        self.piece_y = 0
        if self.check_collision():
            self.game_over = True
    
    def check_collision(self):
        for y in range(self.piece.shape[0]):
            for x in range(self.piece.shape[1]):
                if self.piece[y,x]:
                    bx, by = self.piece_x + x, self.piece_y + y
                    if bx < 0 or bx >= self.width or by >= self.height or (by >= 0 and self.board[by,bx]):
                        return True
        return False
    
    def move(self, dx, dy):
        if self.game_over:
            return False
            
        self.piece_x += dx
        self.piece_y += dy
        if self.check_collision():
            self.piece_x -= dx
            self.piece_y -= dy
            if dy > 0:
                self.place_piece()
                return True
        return False
    
    def place_piece(self):
        for y in range(self.piece.shape[0]):
            for x in range(self.piece.shape[1]):
                if self.piece[y,x]:
                    by, bx = self.piece_y + y, self.piece_x + x
                    if 0 <= by < self.height and 0 <= bx < self.width:
                        self.board[by,bx] = self.colors.index(self.piece_color) + 1
        self.clear_lines()
        self.new_piece()
        self.score += 10
    
    def clear_lines(self):
        full_lines = [i for i, row in enumerate(self.board) if all(row)]
        for line in full_lines:
            self.board = np.delete(self.board, line, axis=0)
            self.board = np.vstack([np.zeros(self.width), self.board])
        self.score += len(full_lines) * 100
    
    def rotate(self):
        if self.game_over:
            return
            
        old_piece = self.piece.copy()
        self.piece = np.rot90(self.piece)
        if self.check_collision():
            self.piece = old_piece

# Create game
game = SimpleTetris()

# Create display
output = widgets.Output()

# Create ultra-compact buttons
def make_btn(text, action, color='lightblue'):
    btn = widgets.Button(
        description=text, 
        layout=widgets.Layout(width='50px', height='35px', margin='1px'),
        style={'button_color': color, 'font_weight': 'bold', 'font_size': '10px'}
    )
    btn.on_click(lambda b: action())
    return btn

def left(): 
    game.move(-1, 0)
    update()

def right(): 
    game.move(1, 0)
    update()

def down(): 
    game.move(0, 1)
    update()

def rotate(): 
    game.rotate()
    update()

def drop(): 
    if not game.game_over:
        while not game.move(0, 1): 
            pass
        update()

def restart():
    game.reset()
    update()

# Create ultra-compact button layout
move_buttons = widgets.HBox([
    make_btn("←", left, 'lightblue'),
    make_btn("↓", down, 'lightgreen'), 
    make_btn("→", right, 'lightblue')
])

action_buttons = widgets.HBox([
    make_btn("↻", rotate, 'yellow'),
    make_btn("↓", drop, 'orange'),  # Changed from bomb to simple arrow
    make_btn("🔄", restart, 'lightcoral')
])

# Compact score display
score_display = widgets.HTML(
    value=f"<div style='text-align: center; font-size: 11px; font-weight: bold; margin: 0;'>Score: {game.score}</div>"
)

def update():
    with output:
        clear_output(wait=True)
        
        # Update score display
        score_display.value = f"<div style='text-align: center; font-size: 11px; font-weight: bold; margin: 0;'>Score: {game.score}</div>"
        
        # Create ultra-compact figure
        fig, ax = plt.subplots(figsize=(2.5, 3))  # Very small figure
        fig.patch.set_facecolor('#2C2C2C')
        
        # Display the board with tiny cells
        cell_size = 0.7  # Even smaller cells
        for y in range(game.height):
            for x in range(game.width):
                if game.board[y, x] > 0:
                    color_idx = int(game.board[y, x]) - 1
                    color = game.colors[color_idx]
                    rect = plt.Rectangle((x, game.height - y - 1), cell_size, cell_size, 
                                       facecolor=color, edgecolor='white', linewidth=0.3)
                    ax.add_patch(rect)
        
        # Draw current piece
        if not game.game_over:
            for y in range(game.piece.shape[0]):
                for x in range(game.piece.shape[1]):
                    if game.piece[y, x]:
                        rect = plt.Rectangle((game.piece_x + x, game.height - (game.piece_y + y) - 1), 
                                           cell_size, cell_size, 
                                           facecolor=game.piece_color, edgecolor='white', linewidth=0.3)
                        ax.add_patch(rect)
        
        # Set up the plot with minimal space
        ax.set_xlim(-0.1, game.width + 0.1)
        ax.set_ylim(-0.1, game.height + 0.1)
        ax.set_aspect('equal')
        ax.set_facecolor('#1E1E1E')
        ax.axis('off')
        
        # Minimal grid
        for x in range(game.width + 1):
            ax.axvline(x, color='gray', linewidth=0.1, alpha=0.1)
        for y in range(game.height + 1):
            ax.axhline(y, color='gray', linewidth=0.1, alpha=0.1)
        
        # Tiny game over message
        if game.game_over:
            ax.text(game.width/2, game.height/2, 'END', 
                   color='red', fontsize=8, fontweight='bold', 
                   ha='center', va='center',
                   bbox=dict(boxstyle="round,pad=0.2", facecolor='black', alpha=0.8))
        
        plt.tight_layout(pad=0.2)  # Minimal padding
        plt.show()

# Create ultra-compact main container
main_container = widgets.VBox([
    widgets.HTML("<div style='text-align: center; font-size: 12px; font-weight: bold; margin: 0; color: white;'>🎮</div>"),
    score_display,
    output,
    widgets.HTML("<div style='text-align: center; font-size: 8px; color: #CCC; margin: 0;'>Controls:</div>"),
    move_buttons,
    action_buttons,
    widgets.HTML("<div style='text-align: center; font-size: 7px; color: #888; margin: 2px 0 0 0;'>Use arrows</div>")
], layout=widgets.Layout(
    align_items='center',
    padding='2px',
    margin='0'
))

# Display everything
display(main_container)
update()

# Auto drop
def auto_drop():
    while True:
        if not game.game_over:
            game.move(0, 1)
            update()
        time.sleep(0.6)

# Start auto-drop thread
drop_thread = threading.Thread(target=auto_drop, daemon=True)
drop_thread.start()

# Add keyboard controls
from IPython.display import Javascript
Javascript("""
document.addEventListener('keydown', function(e) {
    if(['ArrowLeft','ArrowRight','ArrowDown','ArrowUp',' '].includes(e.key)) {
        IPython.notebook.kernel.execute('handle_key("' + e.key + '")');
        e.preventDefault();
    }
});
""")

def handle_key(key):
    if key == 'ArrowLeft':
        left()
    elif key == 'ArrowRight':
        right()
    elif key == 'ArrowDown':
        down()
    elif key == 'ArrowUp':
        rotate()
    elif key == ' ':
        drop()

print("📱 Ultra-compact Tetris ready! Everything should fit perfectly.")